In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import torch
import json
import numpy as np
import pandas as pd

from notebooks.local.utils import get_paths, create_folders

PATHS    = get_paths()
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
THRESHOLDS = [0.05, 0.10, 0.20]

K           = 5
TEMPERATURE = 0.10
MAX_PATCH_DISTS = [0, 1, 2, 3]

create_folders(PATHS)
print("Device:", DEVICE)


In [ ]:
import os
import shutil, tarfile
from notebooks.local.utils import get_paths, create_folders, download_file

PATHS    = get_paths()
create_folders(PATHS)

spair_check = os.path.join(PATHS['spair71k'], 'JPEGImages')
if not os.path.exists(spair_check):
    print('SPair-71k not found. Downloading (~2 GB) ...')
    tar_path = os.path.join(PATHS['data'], 'SPair-71k.tar.gz')
    download_file(
        'http://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz',
        tar_path, desc='SPair-71k',
    )
    print('Extracting ...')
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(PATHS['spair71k'])
    extracted_sub = os.path.join(PATHS['spair71k'], 'SPair-71k')
    if os.path.isdir(extracted_sub):
        for item in os.listdir(extracted_sub):
            shutil.move(os.path.join(extracted_sub, item),
                        os.path.join(PATHS['spair71k'], item))
        os.rmdir(extracted_sub)
    os.remove(tar_path)
    print('SPair-71k ready.')
else:
    print('SPair-71k already present.')

if not os.path.exists(PATHS['dinov2_w']):
    print('Downloading DINOv2 ViT-B/14 weights (~330 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth',
        PATHS['dinov2_w'], desc='DINOv2',
    )
else:
    print('DINOv2 weights present.')

if not os.path.exists(PATHS['sam_w']):
    print('Downloading SAM ViT-B weights (~370 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        PATHS['sam_w'], desc='SAM',
    )
else:
    print('SAM weights present.')

if not os.path.exists(PATHS['dinov3_w']):
    print('WARNING: DINOv3 weights not found at', PATHS['dinov3_w'])
    print('  Place dinov3_vitb16_pretrain.pth in weights/ (obtain from project maintainer).')
else:
    print('DINOv3 weights present.')


In [ ]:
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as vit_base_v2
from src.models.dinov3.dinov3.models.vision_transformer import vit_base as vit_base_v3
from src.datasets.spair_dataset import SPairDataset
from experiments.evaluate import evaluate_with_mnn, save_results


def load_model(backbone, paths, device, use_fp16=False):
    ft_map = {'dinov2': (paths['dinov2_ft'], paths['dinov2_w']),
              'dinov3': (paths['dinov3_ft'], paths['dinov3_w'])}
    ft_path, base_path = ft_map[backbone]

    if backbone == 'dinov2':
        model = vit_base_v2(img_size=(518,518), patch_size=14,
                            num_register_tokens=0, block_chunks=0, init_values=1.0)
        img_size, patch_size = 518, 14
    else:
        model = vit_base_v3(img_size=512, patch_size=16)
        img_size, patch_size = 512, 16

    src = ft_path if os.path.exists(ft_path) else base_path
    ckpt = torch.load(src, map_location=device, weights_only=True)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    model = model.to(device)
    if use_fp16:
        model = model.half()
    model.eval()
    return model, img_size, patch_size


pair_ann = os.path.join(PATHS['spair71k'], 'PairAnnotation')
layout   = os.path.join(PATHS['spair71k'], 'Layout')
images   = os.path.join(PATHS['spair71k'], 'JPEGImages')
test_ds = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'test')
print(f"Test pairs: {len(test_ds)}")


## MNN Ablation — DINOv2

In [ ]:
model, img_size, patch_size = load_model('dinov2', PATHS, DEVICE, USE_FP16)
mnn_results_v2 = {}

for mpd in MAX_PATCH_DISTS:
    out_dir = os.path.join(PATHS['step4_mnn'], f"dinov2_mnn{mpd}")
    os.makedirs(out_dir, exist_ok=True)
    stats_path = os.path.join(out_dir, 'overall_stats.json')

    if os.path.exists(stats_path):
        print(f"DINOv2 MNN max_patch_dist={mpd} — already done.")
    else:
        per_img, all_kp, t = evaluate_with_mnn(
            model, test_ds, DEVICE, THRESHOLDS,
            K=K, temperature=TEMPERATURE, max_patch_dist=mpd,
            use_multilayer=False, use_pca=False,
        )
        save_results(per_img, all_kp, out_dir, t, THRESHOLDS)

    with open(stats_path) as f:
        s = json.load(f)
    mnn_results_v2[mpd] = s
    print(f"DINOv2 MNN max_patch_dist={mpd}: PCK@0.10={s['pck@0.10']['mean']:.2f}%")

del model; torch.cuda.empty_cache()


## MNN Ablation — DINOv3

In [ ]:
model, img_size, patch_size = load_model('dinov3', PATHS, DEVICE, USE_FP16)
mnn_results_v3 = {}

for mpd in MAX_PATCH_DISTS:
    out_dir = os.path.join(PATHS['step4_mnn'], f"dinov3_mnn{mpd}")
    os.makedirs(out_dir, exist_ok=True)
    stats_path = os.path.join(out_dir, 'overall_stats.json')

    if os.path.exists(stats_path):
        print(f"DINOv3 MNN max_patch_dist={mpd} — already done.")
    else:
        per_img, all_kp, t = evaluate_with_mnn(
            model, test_ds, DEVICE, THRESHOLDS,
            K=K, temperature=TEMPERATURE, max_patch_dist=mpd,
            use_multilayer=False, use_pca=False,
        )
        save_results(per_img, all_kp, out_dir, t, THRESHOLDS)

    with open(stats_path) as f:
        s = json.load(f)
    mnn_results_v3[mpd] = s
    print(f"DINOv3 MNN max_patch_dist={mpd}: PCK@0.10={s['pck@0.10']['mean']:.2f}%")

del model; torch.cuda.empty_cache()


## Summary

In [ ]:
rows = []
for backbone, mnn_res in [('dinov2', mnn_results_v2), ('dinov3', mnn_results_v3)]:
    for mpd, s in mnn_res.items():
        rows.append({
            'Model': f"{backbone}",
            'max_patch_dist': mpd,
            'PCK@0.05': round(s.get('pck@0.05',{}).get('mean', float('nan')), 2),
            'PCK@0.10': round(s.get('pck@0.10',{}).get('mean', float('nan')), 2),
            'PCK@0.20': round(s.get('pck@0.20',{}).get('mean', float('nan')), 2),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
